# 02 — Clean & Quality Check
Load raw data into DuckDB, clean it with SQL, run a quality report,
and save the result to `data/interim/` as Parquet.

All cleaning logic lives in SQL inside DuckDB — reproducible, fast, and easy to inspect.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import (
    get_connection, normalize_column_names, clean_table,
    run_sql, quality_report, save_interim, register_source,
)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')


## Load raw data

In [ ]:
# Load the CSV/Parquet saved by 01-ingest.ipynb.
# Adjust the path and read function to match your raw file type.

# df_raw = pd.read_csv("../data/raw/SOURCE_NAME.csv")
# df_raw = normalize_column_names(df_raw)
# print(df_raw.shape)
# df_raw.head()

## Clean with DuckDB
`clean_table()` loads the DataFrame into DuckDB and runs a single SQL pass.
Use `cast_map` to fix dtypes, `strip_columns` for whitespace, and `where_filter` to drop bad rows.

In [ ]:
# df_clean = clean_table(
#     df_raw,
#     table_name="source_raw",
#     con=con,
#     cast_map={"year": "INTEGER", "value": "DOUBLE"},
#     strip_columns=["name", "state"],
#     where_filter="value IS NOT NULL AND value > 0",
#     drop_duplicate_subset=["id"],
# )
# df_clean.head()

## Custom SQL (optional)
Use `run_sql()` for joins, aggregations, or derived columns that are easier in SQL.

In [ ]:
# Example: add a per-capita rate column by joining two tables
# df_enriched = run_sql("""
#     SELECT
#         a.*,
#         ROUND(a.count * 100000.0 / b.population, 2) AS rate_per_100k
#     FROM source_raw a
#     JOIN population_raw b USING (state, year)
# """, con)
# df_enriched.head()

## Quality report

In [ ]:
# qr = quality_report(
#     df_clean,
#     table_name="source_clean",
#     con=con,
#     required_columns=["state", "year", "value"],
#     max_null_pct=0.05,
# )

## Save to interim

In [ ]:
# save_interim(df_clean, cfg, "source_clean.parquet")

---
**Next:** open `03-prepare.ipynb` to build the final analysis-ready dataset and package it for sale.

---
## Cleanup
Close the DuckDB connection so the lock is released for other tools (DBCode, other notebooks). Runs on “Run All”.

In [ ]:
con.close()
print('connection closed')